## HWPX 파싱

In [2]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [3]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]
    practice = sorted(SAMPLES.glob('*.hwpx'))
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')
    print(f'  그중 한글 문서 연습 재료 : {len(practice)}건   ← 오늘 하루 이 셋을 읽습니다')


재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
  그중 한글 문서 연습 재료 : 3건   ← 오늘 하루 이 셋을 읽습니다


In [4]:
from app.rag.local_parsers import parse_docx
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_docx(TARGET)
print(f'블록 수    : {len(doc.blocks)}')
print(f'표 수      : {doc.table_count}')
print(f'page_count : {doc.page_count}')
first = doc.blocks[0]
print(f'첫 블록 kind : {first.kind}')
print(f'첫 블록 text : {first.text}')


블록 수    : 233
표 수      : 7
page_count : 1
첫 블록 kind : 조항
첫 블록 text : 일반


In [5]:
from app.rag.local_parsers import parse_pdf

NAIVE = {'.docx': parse_docx, '.pdf': parse_pdf}

read_ok, skipped = (0, 0)
for p in sorted(SAMPLES.iterdir()):
    if not p.is_file():
        continue
    handler = NAIVE.get(p.suffix.lower())
    if handler is None:
        skipped += 1
        continue
    handler(p)
    read_ok += 1
print(f'훑은 파일   : {read_ok + skipped}건')
print(f'읽힘        : {read_ok}건')
print(f'조용히 빠짐 : {skipped}건')


훑은 파일   : 24건
읽힘        : 20건
조용히 빠짐 : 4건


In [6]:
ZIP_MAGIC = b'PK\x03\x04'
heads = [p.read_bytes()[:4] for p in sorted(SAMPLES.glob('*.hwpx'))]
for n, head in enumerate(heads, 1):
    print(f'연습 재료 {n} : 앞 4바이트 {head!r}  →  ZIP 인가 {head == ZIP_MAGIC}')
print('세 건 모두 ZIP :', all((h == ZIP_MAGIC for h in heads)))

연습 재료 1 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
연습 재료 2 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
연습 재료 3 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
세 건 모두 ZIP : True


In [7]:
import zipfile

HWPX = next(SAMPLES.glob("DOC-HR-014_*_v2.0.hwpx"))
print("가져온 파일: ", HWPX.name)  
print()

with zipfile.ZipFile(HWPX) as zf:
    names = zf.namelist() 

print("항목 수 : ", len(names))
for n in names:
    print(" ", n)

가져온 파일:  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx

항목 수 :  10
  mimetype
  version.xml
  META-INF/container.xml
  META-INF/container.rdf
  META-INF/manifest.xml
  Contents/content.hpf
  Contents/header.xml
  Contents/section0.xml
  settings.xml
  Preview/PrvText.txt


In [8]:
with zipfile.ZipFile(HWPX) as zf:
    raw_mime = zf.read('mimetype')
    info = zf.getinfo('mimetype')
    ways = {zf.getinfo(n).compress_type for n in names}
print('mimetype 내용 :', raw_mime.decode('ascii'))
print('바이트 수     :', len(raw_mime))
print('압축 방식     :', info.compress_type, '(0 = 압축한 것이 아니라 그대로 넣었다)')
print('열 개 전체의 압축 방식 :', ways)


mimetype 내용 : application/hwp+zip
바이트 수     : 19
압축 방식     : 0 (0 = 압축한 것이 아니라 그대로 넣었다)
열 개 전체의 압축 방식 : {0}


In [9]:
import re

with zipfile.ZipFile(HWPX) as zf:
    sections = sorted(n for n in zf.namelist() if re.match(r"Contents/section\d+\.xml$", n))

print("본문파일 : ", sections)
print("개수 : ", len(sections))

본문파일 :  ['Contents/section0.xml']
개수 :  1


In [11]:
with zipfile.ZipFile(HWPX) as zf:
    raw = zf.read(sections[0])

print("type : ", type(raw).__name__)
print("길이 : ", f"{len(raw):,}", "바이트")
print(raw[:200].decode("utf-8"))

type :  bytes
길이 :  190,833 바이트
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<hs:sec xmlns:hs="http://www.hancom.co.kr/hwpml/2011/section" xmlns:hp="http://www.hancom.co.kr/hwpml/2011/paragraph" xmlns:hc="http://www.hanco


In [12]:
from xml.etree import ElementTree as ET
root = ET.fromstring(raw)
print('뿌리 태그 :', root.tag)
print('직계 자식 :', len(root), '개')

뿌리 태그 : {http://www.hancom.co.kr/hwpml/2011/section}sec
직계 자식 : 237 개


In [14]:
MINI = '<sec>\n  <p>제12조(식비)</p>\n  <p><tbl><tc><p>구분</p></tc><tc><p>금액</p></tc></tbl></p>\n</sec>'
mini = ET.fromstring(MINI)
print("findall('p') :", len(mini.findall('p')), '개  ← 바로 아래 자식만')
print("iter('p')    :", len(list(mini.iter('p'))), '개  ← 모든 깊이')
print()
print('findall 이 찾은 것 :', [e.text for e in mini.findall('p')])
print('iter    가 찾은 것 :', [e.text for e in mini.iter('p')])

findall('p') : 2 개  ← 바로 아래 자식만
iter('p')    : 4 개  ← 모든 깊이

findall 이 찾은 것 : ['제12조(식비)', None]
iter    가 찾은 것 : ['제12조(식비)', None, '구분', '금액']


In [15]:
found = root.findall('p')
print("findall('p') 결과 :", found)
print('찾은 개수         :', len(found))
print()
print('에러가 났습니까?  → 아니요. 조용히 빈 목록이 돌아왔습니다.')

findall('p') 결과 : []
찾은 개수         : 0

에러가 났습니까?  → 아니요. 조용히 빈 목록이 돌아왔습니다.


In [16]:
NS_P = '{http://www.hancom.co.kr/hwpml/2011/paragraph}'
tops = root.findall(f'{NS_P}p')
print('최상위 문단 수 :', len(tops))
print('첫 문단의 태그 :', tops[0].tag)

최상위 문단 수 : 237
첫 문단의 태그 : {http://www.hancom.co.kr/hwpml/2011/paragraph}p


In [17]:
try:
    texts = [''.join((t.text for t in para.iter(f'{NS_P}t'))).strip() for para in tops]
    print('모은 문단 :', len(texts))
except TypeError as err:
    print('TypeError:', err)
    print()
    print('→ 문단 237개를 훑다가 중간에서 멈췄습니다. 멈춘 자리 다음 문단들은 아예 시도조차 못 했습니다.')

TypeError: sequence item 0: expected str instance, NoneType found

→ 문단 237개를 훑다가 중간에서 멈췄습니다. 멈춘 자리 다음 문단들은 아예 시도조차 못 했습니다.


In [18]:
def para_text(para) -> str:
	return ''.join((t.text or '' for t in para.iter(f'{NS_P}t'))).strip()
	
texts = [para_text(para) for para in tops]
all_t = list(root.iter(f'{NS_P}t'))

print('문단 수          :', len(texts))
print('글자가 있는 문단 :', sum((1 for s in texts if s)))
print('글자 태그        :', len(all_t), '개')
print('.text 가 None    :', sum((1 for t in all_t if t.text is None)), '개')

문단 수          : 237
글자가 있는 문단 : 233
글자 태그        : 450 개
.text 가 None    : 41 개


In [19]:
by_findall = [s for s in (para_text(p) for p in root.findall(f'{NS_P}p')) if s]

by_iter = [s for s in (para_text(p) for p in root.iter(f'{NS_P}p')) if s]

print(f"{'findall':<8} 로 모은 블록 : {len(by_findall)}")
print(f"{'iter':<8} 로 모은 블록 : {len(by_iter)}")
print(f"{'차이':<8} : +{len(by_iter) - len(by_findall)}")

findall  로 모은 블록 : 233
iter     로 모은 블록 : 416
차이       : +183


In [20]:
from collections import Counter
extra = Counter(by_iter) - Counter(by_findall)
print('늘어난 블록 :', sum(extra.values()), '개')
in_table = set()
for para in root.findall(f'{NS_P}p'):
    for tbl in para.iter(f'{NS_P}tbl'):
        for cell_para in tbl.iter(f'{NS_P}p'):
            s = para_text(cell_para)
            if s:
                in_table.add(s)
print('표 안에 있던 문단 :', len(in_table), '종류')
print('늘어난 것이 전부 표 안 문장인가 :', set(extra) <= in_table)
print()
for s in list(extra)[:3]:
    print('  중복된 예 :', s[:34])


늘어난 블록 : 183 개
표 안에 있던 문단 : 133 종류
늘어난 것이 전부 표 안 문장인가 : True

  중복된 예 : v2.0
  중복된 예 : 버전
  중복된 예 : 구분


In [21]:
from app.integrations.ports import ParsedBlock, ParsedDoc

def read_section(path):
    with zipfile.ZipFile(path) as zf:
        sections = sorted((n for n in zf.namelist() if re.match('Contents/section\\d+\\.xml$', n)))
        return ET.fromstring(zf.read(sections[0]))

def paragraph_blocks(root, locator):
    blocks = []
    for para in root.findall(f'{NS_P}p'):
        text = ''.join((t.text or '' for t in para.iter(f'{NS_P}t'))).strip()
        if text:
            blocks.append(ParsedBlock('조항', locator, text))
    return blocks
print('도우미 둘 준비됨 :', read_section.__name__, '·', paragraph_blocks.__name__)

도우미 둘 준비됨 : read_section · paragraph_blocks


In [22]:
doc_root = read_section(HWPX)
blocks = paragraph_blocks(doc_root, HWPX.stem)
tables = 0
doc = ParsedDoc(blocks=blocks, page_count=1, table_count=tables)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · page_count {doc.page_count}')
print('첫 블록 kind/locator :', doc.blocks[0].kind, '/', doc.blocks[0].locator)
print('두 번째 블록 글자    :', doc.blocks[1].text[:20])

블록 233 · 표 0 · page_count 1
첫 블록 kind/locator : 조항 / DOC-HR-014_국내출장_여비_규정_v2.0
두 번째 블록 글자    : 국내출장 여비 규정


In [23]:
path = ROOT / "backend" / "app" / "rag" / "local_parsers.py"
text = path.read_text(encoding="utf-8")
print(path.name, "·", len(text.splitlines()), "줄")
for name in ['parse_pdf', 'parse_docx', '_table_to_markdown', 'parse_hwpx']:
    print(f"  def {name:<19}:", text.count(f"def {name}("), "개")

local_parsers.py · 92 줄
  def parse_pdf          : 1 개
  def parse_docx         : 1 개
  def _table_to_markdown : 1 개
  def parse_hwpx         : 1 개


In [24]:
import importlib
from app.rag import local_parsers

importlib.reload(local_parsers)

app_doc = local_parsers.parse_hwpx(HWPX)

print(f"블록 : {len(app_doc.blocks)}, 표 : {app_doc.table_count}, page_count : {app_doc.page_count}")


블록 : 233, 표 : 0, page_count : 1


In [25]:
from app.rag.local_parsers import _table_to_markdown
headers = ['서류', '필수 여부', '보존 기간']
rows = [['출장 신청서(승인본)', '필수', '5년'], ['여비 정산서', '필수', '5년']]
print(_table_to_markdown(headers, rows))

| 서류 | 필수 여부 | 보존 기간 |
|---|---|---|
| 출장 신청서(승인본) | 필수 | 5년 |
| 여비 정산서 | 필수 | 5년 |


In [26]:
import zipfile

HWPX = next(SAMPLES.glob("DOC-HR-014_*_v2.0.hwpx"))
with zipfile.ZipFile(HWPX) as zf:                    # HWPX 는 ZIP 이다
    raw = zf.read("Contents/section0.xml").decode("utf-8")

start = raw.find("<hp:tbl")                           # 첫 표가 시작하는 자리
end = raw.find("</hp:tbl>", start) + len("</hp:tbl>")
table_xml = raw[start:end]

print(f"section0.xml 전체 {len(raw):,}자 · 첫 표 {len(table_xml):,}자")
print(table_xml[:800])  

section0.xml 전체 181,123자 · 첫 표 17,952자
<hp:tbl id="1" zOrder="0" numberingType="TABLE" textWrap="TOP_AND_BOTTOM" textFlow="BOTH_SIDES" lock="0" dropcapstyle="None" pageBreak="CELL" repeatHeader="1" rowCnt="4" colCnt="7" cellSpacing="0" borderFillIDRef="2" noAdjust="0"><hp:sz width="44000" widthRelTo="ABSOLUTE" height="4800" heightRelTo="ABSOLUTE" protect="0"/><hp:pos treatAsChar="1" affectLSpacing="0" flowWithText="1" allowOverlap="0" holdAnchorAndSO="0" vertRelTo="PARA" horzRelTo="COLUMN" vertAlign="TOP" horzAlign="LEFT" vertOffset="0" horzOffset="0"/><hp:outMargin left="0" right="0" top="0" bottom="0"/><hp:inMargin left="141" right="141" top="141" bottom="141"/><hp:tr><hp:tc name="" header="1" hasMargin="0" protect="0" editable="0" dirty="0" borderFillIDRef="2"><hp:cellAddr colAddr="0" rowAddr="0"/><hp:cellSpan colSpan="1" ro


In [27]:
import copy
from xml.etree import ElementTree as ET

HP = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"
ET.register_namespace("hp", HP[1:-1])     # 이 줄이 없으면 hp: 대신 ns0: 으로 찍힌다

root = ET.fromstring(raw.encode("utf-8"))
tbl = root.find(f".//{HP}tbl")                        # 첫 표

first_row = copy.deepcopy(tbl.find(f"{HP}tr"))        # 원본을 건드리지 않게 복사
for tc in first_row.findall(f"{HP}tc")[1:]:           # 첫 칸만 남긴다
    first_row.remove(tc)

ET.indent(first_row, space="  ")                      # 들여쓰기 (파이썬 3.9+)
print(ET.tostring(first_row, encoding="unicode"))

<hp:tr xmlns:hp="http://www.hancom.co.kr/hwpml/2011/paragraph">
  <hp:tc name="" header="1" hasMargin="0" protect="0" editable="0" dirty="0" borderFillIDRef="2">
    <hp:cellAddr colAddr="0" rowAddr="0" />
    <hp:cellSpan colSpan="1" rowSpan="1" />
    <hp:cellSz width="6285" height="1200" />
    <hp:cellMargin left="141" right="141" top="141" bottom="141" />
    <hp:subList id="" textDirection="HORIZONTAL" lineWrap="BREAK" vertAlign="CENTER" linkListIDRef="0" linkListNextIDRef="0" textWidth="0" textHeight="0" hasTextRef="0" hasNumRef="0">
      <hp:p id="0" paraPrIDRef="0" styleIDRef="0" pageBreak="0" columnBreak="0" merged="0">
        <hp:run charPrIDRef="1">
          <hp:t>버전</hp:t>
        </hp:run>
      </hp:p>
    </hp:subList>
  </hp:tc>
</hp:tr>


In [28]:
import re
import zipfile
from xml.etree import ElementTree as ET
ns_p = '{http://www.hancom.co.kr/hwpml/2011/paragraph}'
HWPX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.hwpx'))
with zipfile.ZipFile(HWPX) as zf:
    sections = sorted((n for n in zf.namelist() if re.match('Contents/section\\d+\\.xml$', n)))
    root = ET.fromstring(zf.read(sections[0]))
paras = root.findall(f'{ns_p}p')
with_table = [p for p in paras if p.find(f'.//{ns_p}tbl') is not None]
print(f'최상위 문단 {len(paras)}개')
print(f'그중 표를 담은 문단 {len(with_table)}개')

최상위 문단 237개
그중 표를 담은 문단 7개


In [29]:
table_el = with_table[0].find(f'.//{ns_p}tbl')
tr = table_el.findall(f'{ns_p}tr')[0]
tc = tr.findall(f'{ns_p}tc')[0]

def child_names(el):
    """el 의 바로 아래 자식 태그 이름만 모은다 (네임스페이스는 떼고)."""
    return [child.tag.split('}')[-1] for child in el]
print('tbl 직계 :', child_names(table_el)[:5], '…')
print('tr  직계 :', sorted(set(child_names(tr))))
print('tc  직계 :', child_names(tc))


tbl 직계 : ['sz', 'pos', 'outMargin', 'inMargin', 'tr'] …
tr  직계 : ['tc']
tc  직계 : ['cellAddr', 'cellSpan', 'cellSz', 'cellMargin', 'subList']


* 표 문단을 그냥 이으면 칸이 사라진다

In [30]:
# 나쁜 예시
from app.rag.local_parsers import parse_hwpx
doc = parse_hwpx(HWPX)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count}')
first_table_block = next((b for b in doc.blocks if b.text.startswith('버전')))
print('표 자리의 블록 :', first_table_block.text[:60], '…')

블록 233 · 표 0
표 자리의 블록 : 버전구분시행일주요 내용작성검토승인v2.0개정2025-07-01일비 인상(25,000→30,000) · 광역시 …


In [40]:
def table_rows(table_el) -> list[list[str]]:
    rows = []
    for tr in table_el.findall(f'{ns_p}tr'):
        cells = []
        for tc in tr.findall(f'{ns_p}tc'):
            cells.append(''.join((t.text or '' for t in tc.iter(f'{ns_p}t'))).strip())
        rows.append(cells)
    return rows

rows = table_rows(with_table[1].find(f'.//{ns_p}tbl'))
md_table = _table_to_markdown(rows[0], rows[1:])
print(f'{len(rows)}행 × {len(rows[0])}열')
print('\n'.join(md_table.splitlines()[:3]))

5행 × 5열
| 직급 | 일비 | 숙박비(광역시) | 숙박비(그 밖의 지역) | 식비 |
|---|---|---|---|---|
| 1~2급 | 40,000 | 100,000 | 80,000 | 30,000 |


In [41]:
out_path = SANDBOX / 'hwpx_tables_DOC-HR-014_v2.0.md'
parts = []
for n, para in enumerate(with_table, 1):
    r = table_rows(para.find(f'.//{ns_p}tbl'))
    parts.append(f'## 표{n}\n\n' + _table_to_markdown(r[0], r[1:]))
text = '\n\n'.join(parts) + '\n'
out_path.write_text(text, encoding='utf-8')
print('저장 :', out_path.relative_to(ROOT))
print(f'표 {len(parts)}개 · {len(text.splitlines())}줄 · {out_path.stat().st_size} 바이트')


저장 : sandbox/w5/day02/hwpx_tables_DOC-HR-014_v2.0.md
표 7개 · 95줄 · 3549 바이트


In [43]:
def count_blocks(use_continue: bool) -> int:
    n = 0
    for para in root.findall(f'{ns_p}p'):
        if para.find(f'.//{ns_p}tbl') is not None:
            n += 1
            if use_continue:
                continue
        if ''.join((t.text or '' for t in para.iter(f'{ns_p}t'))).strip():
            n += 1
    return n
with_c, without_c = (count_blocks(True), count_blocks(False))
print('continue 있음 :', with_c)
print('continue 없음 :', without_c)
print('차이          :', without_c - with_c)


continue 있음 : 233
continue 없음 : 240
차이          : 7


In [44]:
import importlib
from app.rag import local_parsers

importlib.reload(local_parsers)

doc = local_parsers.parse_hwpx(HWPX)
print(f"블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}")

table_blocks = [b for b in doc.blocks if b.kind == "표"]
print("표 블록 :", [b.locator for b in table_blocks])
print("첫 표 블록의 첫 줄 :", table_blocks[0].text.splitlines()[0])

블록 233 · page_count 1 · table_count 7
표 블록 : ['표1', '표2', '표3', '표4', '표5', '표6', '표7']
첫 표 블록의 첫 줄 : | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |


In [48]:
from app.rag.local_parsers import parse_docx, parse_pdf

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
PDF = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))

results = [('HWPX', local_parsers.parse_hwpx(HWPX)), ('DOCX', parse_docx(DOCX)), ('PDF', parse_pdf(PDF))]
print(f"{'형식':<6}{'블록':>6}{'표':>5}{'쪽수':>6}")

for name, parsed in results:
    print(f'{name:<7}{len(parsed.blocks):>7}{parsed.table_count:>5}{parsed.page_count:>7}')

형식        블록    표    쪽수
HWPX       233    7      1
DOCX       233    7      1
PDF         16    0     16


In [52]:
import tempfile
from app.core.exceptions import ValidationFailed
_CONVERT_HINT = {
	'.hwp': '한글에서 열고 [다른 이름으로 저장] → HWPX 또는 PDF 로 내보내세요. 구 .hwp 는 한컴 독자 바이너리라 열지 않고는 읽을 방법이 없습니다',
	 '.doc': 'Word 에서 열고 .docx 로 저장하세요', 
	 '.ppt': 'PowerPoint 에서 열고 .pptx 로 저장하세요', 
	 '.xls': 'Excel 에서 열고 .xlsx 로 저장하세요', 
	 '.hwt': '한글 서식 파일입니다. 내용이 든 .hwpx 문서를 넣으세요', 
	 '.zip': '압축을 풀고 안의 문서를 한 건씩 넣으세요'
 }
TMP = pathlib.Path(tempfile.mkdtemp())
hwp = TMP / '출장규정_구버전.hwp'
hwp.write_bytes(b'')
for target in [hwp, TMP / '출장규정_구버전.rtf']:
    ext = target.suffix.lower()
    try:
        if ext in _CONVERT_HINT:
            raise ValidationFailed(f'읽을 수 없는 형식입니다: {ext}', detail=_CONVERT_HINT[ext])
        raise ValidationFailed(f'지원하지 않는 형식입니다: {ext}', detail='핸들러를 추가하면 지원할 수 있습니다')
    except ValidationFailed as exc:
        print(f'[{ext}] {exc}')
        print(f'      안내 → {exc.detail}')


[.hwp] 읽을 수 없는 형식입니다: .hwp
      안내 → 한글에서 열고 [다른 이름으로 저장] → HWPX 또는 PDF 로 내보내세요. 구 .hwp 는 한컴 독자 바이너리라 열지 않고는 읽을 방법이 없습니다
[.rtf] 지원하지 않는 형식입니다: .rtf
      안내 → 핸들러를 추가하면 지원할 수 있습니다


In [53]:
SCANNED = SAMPLES / '_formats' / 'sample_scanned.pdf'
doc = parse_pdf(SCANNED)
print(f'{SCANNED.name} : 블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')
print('예외가 났는가 : 아니오 (여기까지 왔다는 것이 곧 증거다)')
print()
try:
    if not doc.blocks:
        raise ValidationFailed(f'문서에서 글자를 찾지 못했습니다: {SCANNED.name}', detail='스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요')
except ValidationFailed as exc:
    print('막았습니다 :', exc)
    print('안내       :', exc.detail)


14:46:29 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
sample_scanned.pdf : 블록 0 · 표 0 · 쪽 1
예외가 났는가 : 아니오 (여기까지 왔다는 것이 곧 증거다)

막았습니다 : 문서에서 글자를 찾지 못했습니다: sample_scanned.pdf
안내       : 스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요


In [54]:
from app.core.logging import get_logger

log = get_logger(__name__)

broken = TMP / "broken.hwpx"
broken.write_bytes(b"")

try:
    result = parse_hwpx(broken)
except Exception as e:
    log.warning("로컬 파싱 실패 (%s) : %s", broken.name, e)
    result = None

print(result)

14:50:50 WARNING  __main__ : 로컬 파싱 실패 (broken.hwpx) : File is not a zip file
None
